# **03. Lọc Spam và Định lượng Độ relevancy (Review Filtering)**

## **Mục tiêu:**
- Thiết lập quy trình lọc nhiễu nâng cao:
  - **Độ bất định thông tin (Shannon Entropy Filter)** để loại bỏ các bình luận vô nghĩa dạng gõ phím vô định (ví dụ: `aaaaaaa`, `hdhshshd`).
  - **Độ dài từ ngữ (Length Filter)** loại bỏ bình luận quá ngắn không mang cảm xúc rõ ràng.
  - **Mật độ từ vựng liên quan (Valid Keyword Ratio Filter)** đảm bảo bình luận chứa các từ ngữ liên quan đến sản phẩm/mua sắm và dịch vụ.

In [ ]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath('../src'))
import utils
from filtering import apply_filters

### **1. Nạp dữ liệu đã tiền xử lý**

In [ ]:
df = pd.read_csv("../data/processed/clean_reviews.csv", encoding="utf-8-sig")
print(f"Đang kiểm định lọc trên {len(df):,} dòng bình luận.")

### **2. Áp dụng quy trình lọc**
Dùng module `filtering` để tính toán trạng thái hợp lệ và lý do từ chối.

In [ ]:
print("Applying filtering pipeline...")
# Chạy apply_filters trả về (is_valid, reason)
results = df["cleaned_comment"].astype(str).apply(apply_filters)

df["is_valid"] = [r[0] for r in results]
df["reject_reason"] = [r[1] for r in results]

print("\n--- THỐNG KÊ LÝ DO PHÂN LOẠI ---")
reason_counts = df["reject_reason"].value_counts()
print(reason_counts)

### **3. Thống kê chi tiết tỷ lệ lọc**
Tính toán xem bao nhiêu % bình luận bị loại bỏ ở mỗi bước.

In [ ]:
total_reviews = len(df)
valid_count = df["is_valid"].sum()
spam_count = total_reviews - valid_count

print(f"📊 Kết quả lọc dữ liệu:")
print(f"   + Tổng số bình luận đầu vào: {total_reviews:,} dòng")
print(f"   + Bình luận HỢP LỆ (Giữ lại): {valid_count:,} dòng ({valid_count/total_reviews*100:.2f}%)")
print(f"   + Bình luận RÁC/SPAM (Loại bỏ): {spam_count:,} dòng ({spam_count/total_reviews*100:.2f}%)")

print("\nMẫu các bình luận bị loại bỏ vì lý do Shannon Entropy (Spam nhận xu):")
display(df[df["reject_reason"] == "entropy"][["comment", "cleaned_comment"]].head(5))

print("\nMẫu các bình luận bị loại bỏ vì lý do Keyword Ratio (Thiếu từ liên quan):")
display(df[df["reject_reason"] == "keyword_ratio"][["comment", "cleaned_comment"]].head(5))

### **4. Lưu tập bình luận hợp lệ**
Lưu lại tập dữ liệu đã lọc sạch sẽ.

In [ ]:
valid_df = df[df["is_valid"]].copy()
valid_df.to_csv("../data/processed/valid_reviews.csv", index=False, encoding="utf-8-sig")
print(f"✅ Đã xuất tập {len(valid_df):,} bình luận hợp lệ sang ../data/processed/valid_reviews.csv")